In [19]:
import numpy as np
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)
from datasets import Dataset
import sacrebleu

In [8]:
pip install sacremoses

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [24]:
HING_MODEL = "Helsinki-NLP/opus-mt-hi-en"
SPAN_MODEL = "Helsinki-NLP/opus-mt-es-en"

MAX_LENGTH = 64
BATCH_SIZE = 4
GRAD_ACCUM = 4

HING_EPOCHS = 20
SPAN_EPOCHS = 10

FORCE_CPU = True


def get_device():
    if not FORCE_CPU and torch.cuda.is_available():
        return "cuda"
    if not FORCE_CPU and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


device = get_device()
print("Using device:", device)

Using device: cpu


In [21]:
print("Loading data...")
hing_train = pd.read_csv("data/hinglish_train.csv")
hing_val = pd.read_csv("data/hinglish_val.csv")
hing_test = pd.read_csv("data/hinglish_test.csv")

span_train = pd.read_csv("data/spanglish_train.csv")
span_val = pd.read_csv("data/spanglish_val.csv")
span_test = pd.read_csv("data/spanglish_test.csv")

print(f"Hinglish: {len(hing_train)} train, {len(hing_val)} val, {len(hing_test)} test")
print(f"Spanglish: {len(span_train)} train, {len(span_val)} val, {len(span_test)} test")


Loading data...
Hinglish: 743 train, 93 val, 93 test
Spanglish: 844 train, 105 val, 106 test


In [22]:
def tokenize_df(df, tokenizer, prefix=""):
    src = [(prefix + t) for t in df["source"].astype(str)]
    tgt = df["target"].astype(str).tolist()

    enc = tokenizer(src, max_length=MAX_LENGTH, truncation=True, padding=False)
    dec = tokenizer(tgt, max_length=MAX_LENGTH, truncation=True, padding=False)

    return Dataset.from_dict(
        {
            "input_ids": enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "labels": dec["input_ids"]
        }
    )

def compute_metrics(eval_pred, tokenizer):
    preds, labels = eval_pred

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    bleu = sacrebleu.corpus_bleu(decoded_preds, [decoded_labels]).score
    chrf = sacrebleu.corpus_chrf(decoded_preds, [decoded_labels]).score

    return {"bleu": bleu, "chrf": chrf}


def evaluate_model(path, test_df, tokenizer):
    model = AutoModelForSeq2SeqLM.from_pretrained(path).to(device)
    model.eval()

    refs = test_df["target"].astype(str).tolist()
    preds = []

    batch = 32
    src = test_df["source"].astype(str).tolist()

    for i in range(0, len(src), batch):
        chunk = src[i:i+batch]
        enc = tokenizer(chunk, return_tensors="pt", padding=True,
                        truncation=True, max_length=MAX_LENGTH).to(device)

        with torch.no_grad():
            out = model.generate(**enc, max_length=MAX_LENGTH, num_beams=4)

        preds.extend(tokenizer.batch_decode(out, skip_special_tokens=True))

    preds = [p.strip() for p in preds]
    bleu = sacrebleu.corpus_bleu(preds, [refs]).score
    chrf = sacrebleu.corpus_chrf(preds, [refs]).score
    em = 100.0 * sum(p.lower() == r.lower() for p, r in zip(preds, refs)) / len(refs)

    return preds, bleu, chrf, em

In [23]:
print("TRAINING HINGLISH MODEL")


tokenizer_h = AutoTokenizer.from_pretrained(HING_MODEL)
model_h = AutoModelForSeq2SeqLM.from_pretrained(HING_MODEL).to(device)

train_ds = tokenize_df(hing_train, tokenizer_h, prefix="translate Hinglish to English: ")
val_ds   = tokenize_df(hing_val, tokenizer_h, prefix="translate Hinglish to English: ")

collator = DataCollatorForSeq2Seq(tokenizer=tokenizer_h, model=model_h)

hing_args = Seq2SeqTrainingArguments(
    output_dir="models/marian_hinglish",
    num_train_epochs=HING_EPOCHS,
    learning_rate=1e-4,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    eval_strategy="epoch",            # << fix here
    save_strategy="epoch",
    save_total_limit=2,
    logging_steps=50,
    predict_with_generate=True,
    fp16=False,
    report_to="none",
    no_cuda=(device != "cuda"),
)

hing_trainer = Seq2SeqTrainer(
    model=model_h,
    args=hing_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    tokenizer=tokenizer_h,
    compute_metrics=lambda x: compute_metrics(x, tokenizer_h),
)

hing_trainer.train()
hing_trainer.save_model("models/marian_hinglish/best_model")
tokenizer_h.save_pretrained("models/marian_hinglish/best_model")


TRAINING HINGLISH MODEL


/Users/chidipothusiritha/Library/Python/3.9/lib/python/site-packages/transformers/training_args.py:1636: FutureWarning: using `no_cuda` is deprecated and will be removed in version 5.0 of 🤗 Transformers. Use `use_cpu` instead
  warnings.warn(
/var/folders/yq/pgkldf4s0ng9v2h3wf5m0chw0000gn/T/ipykernel_44944/2792398823.py:28: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  hing_trainer = Seq2SeqTrainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Epoch,Training Loss,Validation Loss,Bleu,Chrf
1,No log,3.447662,3.006406,20.811178
2,4.309500,3.035725,5.252545,24.855096
3,2.881300,2.838553,8.560468,29.495925
4,2.204300,2.749785,8.222781,30.092836
5,1.731000,2.689384,9.777909,31.111481
6,1.374600,2.716130,11.001921,32.642403
7,1.059400,2.679466,11.153993,32.514815
8,0.844300,2.727301,11.846814,33.071413
9,0.656900,2.727247,12.153131,34.830007
10,0.490100,2.752582,12.315065,33.971012


/Users/chidipothusiritha/Library/Python/3.9/lib/python/site-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 512, 'num_beams': 6, 'bad_words_ids': [[61126]]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


('models/marian_hinglish/best_model/tokenizer_config.json',
 'models/marian_hinglish/best_model/special_tokens_map.json',
 'models/marian_hinglish/best_model/vocab.json',
 'models/marian_hinglish/best_model/source.spm',
 'models/marian_hinglish/best_model/target.spm',
 'models/marian_hinglish/best_model/added_tokens.json')

In [26]:
print("TRAINING SPANGLISH MODEL")


tokenizer_s = AutoTokenizer.from_pretrained(SPAN_MODEL)
model_s = AutoModelForSeq2SeqLM.from_pretrained(SPAN_MODEL).to(device)

train_ds = tokenize_df(span_train, tokenizer_s)
val_ds = tokenize_df(span_val, tokenizer_s)

collator = DataCollatorForSeq2Seq(tokenizer=tokenizer_s, model=model_s)

span_args = Seq2SeqTrainingArguments(
    output_dir="models/marian_spanglish",
    num_train_epochs=SPAN_EPOCHS,
    learning_rate=1e-5,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    eval_strategy="epoch",           # << fix
    save_strategy="epoch",
    save_total_limit=2,
    logging_steps=50,
    predict_with_generate=True,
    fp16=False,
    report_to="none",
    no_cuda=(device != "cuda"),
)

span_trainer = Seq2SeqTrainer(
    model=model_s,
    args=span_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    tokenizer=tokenizer_s,
    compute_metrics=lambda x: compute_metrics(x, tokenizer_s),
)

span_trainer.train()
span_trainer.save_model("models/marian_spanglish/best_model")
tokenizer_s.save_pretrained("models/marian_spanglish/best_model")






TRAINING SPANGLISH MODEL


/Users/chidipothusiritha/Library/Python/3.9/lib/python/site-packages/transformers/training_args.py:1636: FutureWarning: using `no_cuda` is deprecated and will be removed in version 5.0 of 🤗 Transformers. Use `use_cpu` instead
  warnings.warn(
/var/folders/yq/pgkldf4s0ng9v2h3wf5m0chw0000gn/T/ipykernel_44944/1140431505.py:28: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  span_trainer = Seq2SeqTrainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Epoch,Training Loss,Validation Loss,Bleu,Chrf
1,2.995800,2.243374,58.936889,77.469189
2,2.172400,1.861214,55.651005,75.298424
3,1.861700,1.661800,53.094990,73.628957
4,1.668300,1.542985,52.754791,73.250865
5,1.520900,1.463491,53.304278,72.665188
6,1.442800,1.413647,53.237489,72.499815
7,1.388300,1.381387,53.262442,72.351339
8,1.313600,1.358812,53.521966,72.432993
9,1.276200,1.346462,53.054148,71.987119
10,1.281100,1.342086,52.538536,71.636433


/Users/chidipothusiritha/Library/Python/3.9/lib/python/site-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 512, 'num_beams': 4, 'bad_words_ids': [[65000]]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


('models/marian_spanglish/best_model/tokenizer_config.json',
 'models/marian_spanglish/best_model/special_tokens_map.json',
 'models/marian_spanglish/best_model/vocab.json',
 'models/marian_spanglish/best_model/source.spm',
 'models/marian_spanglish/best_model/target.spm',
 'models/marian_spanglish/best_model/added_tokens.json')

In [27]:
print("\n" + "="*70)
print("EVALUATING MODELS")
print("="*70)

print("\nHinglish Evaluation:")
hing_preds, hing_bleu, hing_chrf, hing_em = evaluate_model(
    "models/marian_hinglish/best_model", hing_test, tokenizer_h
)
print(f"BLEU={hing_bleu:.2f}, chrF={hing_chrf:.2f}, EM={hing_em:.2f}%")

print("\nSpanglish Evaluation:")
span_preds, span_bleu, span_chrf, span_em = evaluate_model(
    "models/marian_spanglish/best_model", span_test, tokenizer_s
)
print(f"BLEU={span_bleu:.2f}, chrF={span_chrf:.2f}, EM={span_em:.2f}%")

hing_test["marian_prediction"] = hing_preds
span_test["marian_prediction"] = span_preds

hing_test.to_csv("hinglish_marian_results.csv", index=False)
span_test.to_csv("spanglish_marian_results.csv", index=False)

print("\n Models trained and results saved.")


EVALUATING MODELS

Hinglish Evaluation:
BLEU=10.19, chrF=32.35, EM=2.15%

Spanglish Evaluation:
BLEU=46.12, chrF=64.40, EM=2.83%

 Models trained and results saved.


In [30]:
import random

def show_examples(model_path, tokenizer, test_df, num_examples=5):
    model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to(device)
    model.eval()

    print("\n" + "="*60)
    print(f" Sample Translations ({model_path})")
    print("="*60)

    samples = test_df.sample(num_examples)

    for _, row in samples.iterrows():
        src = row["source"]
        tgt = row["target"]

        encoded = tokenizer(
            src,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH
        ).to(device)

        with torch.no_grad():
            output = model.generate(**encoded, max_length=MAX_LENGTH, num_beams=6)

        pred = tokenizer.decode(output[0], skip_special_tokens=True)

        print("\n Source:", src)
        print("Target (reference):", tgt)
        print("Model Prediction:", pred)


In [ ]:
# Show Hinglish examples
show_examples(
    "models/marian_hinglish/best_model",
    tokenizer_h,
    hing_test,
    num_examples=5
)



 Sample Translations (models/marian_hinglish/best_model)

 Source: Hai kya?
Target (reference): Is it?
Model Prediction: Yes?

 Source: mein bhi maantha hoon... voh apni emotions ache tarah hold karti.. lekin dhoda dull lagi
Target (reference): I agree. It seems like she holds back on her emotions. Very dull
Model Prediction: I was also really emotions. I can see why it led to Batman dull

 Source: AAP MUJSE YEH BAAT BOLNA CHAHIYE THA ...LOL
Target (reference): MAYBE YOU SHOULD BE TELLING ME ABOUT IT...LOL
Model Prediction: I like how we tell when we are done

 Source: Elsa queen bani hoti hein, lekin voh apni powers ke baare mein logon ko patha nahi chalna chahiye ke dar bahut tha. jab castle party mein uska choti bahen Anna ko us visiting dignitary ke saath love hotha hein, jo use propose kartha hein, elsa neh toh no bole aur apni power ko saare logon ke saamne dekha dethi hein
Target (reference): Elsa eventually becomes queen, but she's terrified that people will find out about her

In [32]:
# Show Spanglish examples
show_examples(
    "models/marian_spanglish/best_model",
    tokenizer_s,
    span_test,
    num_examples=5
)


 Sample Translations (models/marian_spanglish/best_model)

 Source: Y obtuvimos, de nuevo, about three to six times más stem cells que con el procedimiento standard en el mismo paciente.
Target (reference): And we got, again, about three to six times more stem cells than the standard approach done on the same patient.
Model Prediction: And we getted, are, about three to six times more stem cells that with the standard procedure in the same patient.

 Source: De ese modo create uniform lighting de una pared a la otra in a regular grid of lámparas.
Target (reference): This is how we create uniform lighting from one wall to the other in a regular grid of lamps.
Model Prediction: So create uniform lighting from one wall to the other in a regular grid of lumens.

 Source: "Voy a hablar de dos architects very, very briefly que representan el current split, arquitectónicamente, entre estas dos traditions of a technocratic or technological solution y la solución romántica."
Target (reference)